# Retrieve and Clean Student CSV Datasets

This notebook loads the yearly student CSV files from `DataSets/`, standardizes their columns, combines them into a single dataset, and produces a cleaned version.

Run the cells in order from top to bottom. It assumes the notebook's working directory is `Scripts/`.

In [1]:
import csv
import json
from pathlib import Path
import pandas as pd
import numpy as np
from joblib import dump
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

BASE_DIR = Path.cwd().resolve().parent
DATASETS_DIR = BASE_DIR / "DataSets"

## Helper Functions

Discover CSV files, detect their delimiter, and standardize column names.

In [2]:
# Only the per-year extracts are inputs. Globbing "*.csv" would also pick up this
# notebook's own outputs and fold them back into the next run.
YEARLY_PATTERN = "datos_estudiantes_[0-9][0-9][0-9][0-9].csv"
COMBINED_NAME = "datos_estudiantes_total.csv"
CLEAN_NAME = "datos_estudiantes_total_clean.csv"

# The 2025 extract renames three CREA measures with an "en CREA" suffix while earlier
# years use the unsuffixed names. Mapping them onto one name keeps the combined frame at
# one column per measure; without this, each name is missing for the years that use the
# other spelling, and those gaps look like real data once anything fills them in.
COLUMN_ALIASES = {
    "cantidad_de_entregas_de_tareas_en_crea": "cantidad_de_entregas_de_tareas",
    "cantidad_de_comentarios_posteados_en_crea": "cantidad_de_comentarios_posteados",
    "cantidad_de_acciones_totales_en_crea": "cantidad_de_acciones_totales",
}


def get_csv_files():
    csv_files = sorted(DATASETS_DIR.glob(YEARLY_PATTERN))

    if not csv_files:
        print(f"No yearly CSV files found in {DATASETS_DIR}")
        return []

    return csv_files


def detect_delimiter(path: Path):
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        sample = file.read(4096)
        file.seek(0)

    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=",;")
        return dialect.delimiter
    except csv.Error:
        return ";" if ";" in sample else ","


def standardize_column_name(column_name):
    standardized = str(column_name).strip().lower().replace(" ", "_")
    return COLUMN_ALIASES.get(standardized, standardized)


def get_standardized_columns(csv_file: Path):
    delimiter = detect_delimiter(csv_file)

    with csv_file.open("r", encoding="utf-8-sig", newline="") as file:
        reader = csv.reader(file, delimiter=delimiter)
        header = next(reader, None)

    if header is None:
        return []

    standardized = [standardize_column_name(column) for column in header]
    return standardized


def load_csv_as_dataframe(csv_file: Path):
    delimiter = detect_delimiter(csv_file)
    df = pd.read_csv(csv_file, sep=delimiter, encoding="utf-8-sig")
    df.columns = [standardize_column_name(col) for col in df.columns]
    return df

## Preview Raw Files

Print the columns and first rows of each raw CSV file.

In [3]:
def print_first_rows(csv_file: Path, rows_to_show: int = 5):
    delimiter = detect_delimiter(csv_file)

    with csv_file.open("r", encoding="utf-8-sig", newline="") as file:
        reader = csv.reader(file, delimiter=delimiter)
        rows = list(reader)

    if not rows:
        print(f"\n=== {csv_file.name} ===")
        print("File is empty.")
        return

    print(f"\n=== {csv_file.name} ===")
    print("Columns:", get_standardized_columns(csv_file))
    for row in rows[:rows_to_show]:
        print(row)


csv_files = get_csv_files()

for csv_file in csv_files:
    print_first_rows(csv_file)


=== datos_estudiantes_2019.csv ===
Columns: ['id_persona', 'sexo', 'rol', 'departamento', 'subsistema', 'ciclo', 'grado', 'zona', 'contexto', 'año_lectivo', 'cantidad_de_días_ingreso_a_crea', 'cantidad_de_entregas_de_tareas', 'cantidad_de_comentarios_posteados', 'cantidad_de_acciones_totales', 'cantidad_de_días_de_ingreso_a_matific', 'cantidad_de_episodios_finalizados_en_matific', 'cantidad_de_días_de_ingreso_a_pam', 'cantidad_de_actividades_finalizadas_en_pam', 'cantidad_de_días_de_ingreso_a_biblioteca', 'cantidad_de_préstamos_en_biblioteca']
['Id persona', 'Sexo', 'Rol', 'Departamento', 'Subsistema', 'Ciclo', 'Grado', 'Zona', 'Contexto', 'Año lectivo', 'Cantidad de días ingreso a CREA', 'Cantidad de entregas de tareas', 'Cantidad de Comentarios posteados', 'Cantidad de Acciones totales', 'Cantidad de días de ingreso a Matific', 'Cantidad de episodios finalizados en Matific', 'Cantidad de días de ingreso a PAM', 'Cantidad de actividades finalizadas en PAM', 'Cantidad de días de ingre

## Combine Datasets

Load every yearly CSV, standardize columns, concatenate, drop duplicates, and save the combined dataset.

> Note: the combined and cleaned outputs are large (500MB+) and are excluded from git via `.gitignore`.

In [4]:
def report_coverage(combined: pd.DataFrame):
    """Show the observed fraction of each column per year to expose schema drift."""
    coverage = (
        combined.notna().groupby(combined["año_lectivo"]).mean().round(3).sort_index().T
    )
    print("\nObserved fraction by year (0.0 means the year lacks the column):")
    print(coverage.to_string())
    return coverage


def create_combined_dataset(output_name: str = COMBINED_NAME):
    csv_files = get_csv_files()
    if not csv_files:
        return None

    dataframes = [load_csv_as_dataframe(csv_file) for csv_file in csv_files]
    combined = pd.concat(dataframes, ignore_index=True)
    combined = combined.drop_duplicates()

    output_path = DATASETS_DIR / output_name
    combined.to_csv(output_path, index=False)

    print(f"Combined dataset saved to: {output_path}")
    print(f"Rows: {len(combined)}")
    print(f"Columns: {list(combined.columns)}")
    report_coverage(combined)
    return combined


combined = create_combined_dataset()

Combined dataset saved to: /Users/gerardo/Documents/GitHub/PlanCeibal-UTEC26-MachineLearning/DataSets/datos_estudiantes_total.csv
Rows: 4560441
Columns: ['id_persona', 'sexo', 'rol', 'departamento', 'subsistema', 'ciclo', 'grado', 'zona', 'contexto', 'año_lectivo', 'cantidad_de_días_ingreso_a_crea', 'cantidad_de_entregas_de_tareas', 'cantidad_de_comentarios_posteados', 'cantidad_de_acciones_totales', 'cantidad_de_días_de_ingreso_a_matific', 'cantidad_de_episodios_finalizados_en_matific', 'cantidad_de_días_de_ingreso_a_pam', 'cantidad_de_actividades_finalizadas_en_pam', 'cantidad_de_días_de_ingreso_a_biblioteca', 'cantidad_de_préstamos_en_biblioteca']

Observed fraction by year (0.0 means the year lacks the column):
año_lectivo                                    2019   2020  2021   2022   2023   2024   2025
id_persona                                    1.000  1.000  1.00  1.000  1.000  1.000  1.000
sexo                                          1.000  1.000  1.00  1.000  1.000  1.000  1.

## Clean the Combined Dataset

Normalize text, replace known "missing" placeholders with NA, and coerce numeric-looking columns. Missing values are preserved so model preprocessing can be fit on training data only.

In [ ]:
def resolve_id_year_conflicts(df: pd.DataFrame) -> pd.DataFrame:
    """Merge or discard rows with duplicate id_persona+año_lectivo.

    Rows identical on identity fields (sexo, departamento, etc.) are merged by
    summing activity metrics. Rows with conflicting identity are discarded.
    """
    IDENTITY_COLUMNS = ["sexo", "departamento", "subsistema", "ciclo", "grado", "zona", "contexto"]
    ACTIVITY_COLUMNS = [
        "cantidad_de_días_ingreso_a_crea",
        "cantidad_de_entregas_de_tareas",
        "cantidad_de_comentarios_posteados",
        "cantidad_de_acciones_totales",
        "cantidad_de_días_de_ingreso_a_matific",
        "cantidad_de_episodios_finalizados_en_matific",
        "cantidad_de_días_de_ingreso_a_pam",
        "cantidad_de_actividades_finalizadas_en_pam",
        "cantidad_de_días_de_ingreso_a_biblioteca",
        "cantidad_de_préstamos_en_biblioteca",
    ]

    key = ["id_persona", "año_lectivo"]
    conflict_mask = df.duplicated(subset=key, keep=False)

    if not conflict_mask.any():
        return df

    stable = df[~conflict_mask].copy()
    conflicts = df[conflict_mask]

    merged_rows, dropped = [], 0
    for _, group in conflicts.groupby(key):
        identity_is_consistent = all(
            group[col].dropna().nunique() <= 1 for col in IDENTITY_COLUMNS if col in group.columns
        )
        if identity_is_consistent:
            row = group.iloc[0].copy()
            for col in ACTIVITY_COLUMNS:
                if col in group.columns:
                    row[col] = group[col].sum(min_count=1)
            merged_rows.append(row)
        else:
            dropped += len(group)

    result = pd.concat([stable, pd.DataFrame(merged_rows)], ignore_index=True)
    print(f"id+año conflicts: {len(merged_rows)} pares merged, {dropped} rows discarded (identity mismatch)")
    return result


def add_outlier_flags(df: pd.DataFrame) -> pd.DataFrame:
    """Flag activity metrics above p99.5 percentile."""
    ACTIVITY_COLUMNS = [
        "cantidad_de_comentarios_posteados",
        "cantidad_de_acciones_totales",
        "cantidad_de_actividades_finalizadas_en_pam",
    ]

    for column in ACTIVITY_COLUMNS:
        if column in df.columns:
            threshold = df[column].quantile(0.995)
            df[f"{column}_outlier"] = df[column] > threshold

    return df


def normalize_grado_by_subsistema(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize grado notation by subsistema for consistency.
    
    DGEIP (primaria): 1º, 2º, ..., 6º → 1_p, 2_p, ..., 6_p (remove º, add suffix)
    DGES (ciclo básico): 1-9 → 1_c, 2_c, ..., 9_c (add suffix for clarity)
    DGETP (ed. media técnica): 1ero. → 1_t, 1 → 1_t, etc. (normalize abbreviation, add suffix)
    Special ed categories (sordos, discapacidad, etc.) kept as-is.
    """
    df = df.copy()
    
    def map_grado(row):
        grado_str = str(row['grado']).strip()
        subsistema = str(row['subsistema']).strip().lower()
        
        if pd.isna(row['grado']):
            return pd.NA
        
        if subsistema == 'dgeip':
            grado_clean = grado_str.replace('º', '').strip()
            if grado_clean in ['1', '2', '3', '4', '5', '6']:
                return f"{grado_clean}_p"
            else:
                return grado_str  # Keep special ed as-is
        elif subsistema == 'dges':
            if grado_str in ['1', '2', '3', '4', '5', '6', '7', '8', '9']:
                return f"{grado_str}_c"
            else:
                return grado_str
        elif subsistema == 'dgetp':
            grado_clean = grado_str.replace('ero.', '').strip()
            if grado_clean in ['0', '1', '2', '3', '4', '7', '8', '9']:
                return f"{grado_clean}_t"
            else:
                return grado_str
        else:
            return grado_str
    
    df['grado'] = df.apply(map_grado, axis=1)
    print("\nGRADO normalized by subsistema. Top 25 values post-normalization:")
    print(df['grado'].value_counts().head(25))
    return df


def clean_combined_dataset(df: pd.DataFrame):
    cleaned = df.copy()
    cleaned = cleaned.drop_duplicates()

    for column in cleaned.columns:
        if cleaned[column].dtype == "object":
            cleaned[column] = cleaned[column].astype(str).str.strip().str.lower()
            cleaned[column] = cleaned[column].replace(
                {
                    "": pd.NA,
                    "na": pd.NA,
                    "n/a": pd.NA,
                    "nan": pd.NA,
                    "null": pd.NA,
                    "none": pd.NA,
                    "sin dato": pd.NA,
                    "sin datos": pd.NA,
                    "sin clasificar": pd.NA,
                    "sin_dato": pd.NA,
                    "sin-dato": pd.NA,
                    "unknown": pd.NA,
                    "desconocido": pd.NA,
                }
            )

    if "id_persona" in cleaned.columns:
        cleaned = cleaned.dropna(subset=["id_persona"])

    for column in cleaned.columns:
        if cleaned[column].dtype == "object":
            numeric_values = pd.to_numeric(cleaned[column], errors="coerce")
            valid_ratio = numeric_values.notna().sum() / max(cleaned[column].notna().sum(), 1)
            if valid_ratio > 0.8:
                cleaned[column] = numeric_values

    for column in cleaned.columns:
        if pd.api.types.is_numeric_dtype(cleaned[column]):
            if cleaned[column].dropna().mod(1).eq(0).all():
                cleaned[column] = cleaned[column].astype("Int64")

    cleaned = resolve_id_year_conflicts(cleaned)
    cleaned = add_outlier_flags(cleaned)
    cleaned = normalize_grado_by_subsistema(cleaned)

    # Deliberately no imputation: a year that never reported a measure must stay NA.
    # Filling those gaps invents a constant that later reads as a real observation.
    return cleaned